# CS781 Phase 3 — Rhetorical Role Prediction (Model 2: gated linear-recurrent LM, `ours_glru*`)
Starting point: our own Phase-1 gated-linear-recurrence legal LM (no external weights). Each sentence gets one of 13 rhetorical roles; CL and IT are trained jointly and evaluated separately. Experiments in this notebook:
1. **LoRMA+** (frozen backbone, multiplicative low-rank adaptation) with **context-aware** input (windows of consecutive sentences)
2. **Knowledge distillation** — half-depth student (initialised from Phase-1 blocks) learns from the LoRMA+ teacher's sentence-level logits, full fine-tuning
3. **LoRA** (frozen backbone, additive low-rank adaptation), context-aware: the additive-vs-multiplicative comparison
4. **LoRMA+ with sentence-only input** (ablation: does document context help?)

Sentence representations are mean-pooled token states from a new bidirectional attention head block, followed by a linear classifier. Each experiment writes a `prediction.json` (one label per sentence) and a zip.

## 1. Setup and imports

In [1]:
import os
import sys
import json
import glob
import time
import math
import pickle
import random
import zipfile
import subprocess

import numpy as np


def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args])


try:
    import regex
except ImportError:
    pip_install("regex")
    import regex

gpu_check = "import jax, sys; sys.exit(jax.devices()[0].platform not in ('gpu', 'cuda'))"
if subprocess.run([sys.executable, "-c", gpu_check]).returncode != 0:
    pip_install("-U", "jax[cuda12]")

import jax
import jax.numpy as jnp
from jax import lax
from huggingface_hub import hf_hub_download, login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF"))
except Exception:
    pass

T0 = time.time()
random.seed(0)
np.random.seed(0)
tmap = jax.tree_util.tree_map
print("devices:", jax.devices())

devices: [CudaDevice(id=0), CudaDevice(id=1)]


## 2. Hyperparameters and time budget

In [2]:
CONTEXT_TOKENS = 512
SENT_TOKENS = 128
MAX_SENTS = 48
TRAIN_TOKENS = 4096
PRED_TOKENS = 8192
RANK = 16
PEFT_ALPHA = 16
PEFT_SCALE = PEFT_ALPHA / RANK
KD_TEMP = 2.0
KD_ALPHA = 0.5
MAX_EPOCHS = 30
PATIENCE = 6
WARMUP_STEPS = 100
BIG = 1 << 30
EOS = 3
MAX_SECONDS = 7.5 * 3600
LORMA_MINUTES = 80
KD_MINUTES = 70
LORA_MINUTES = 70
SENTENCE_MINUTES = 50


def budget(minutes):
    return min(minutes * 60, MAX_SECONDS - (time.time() - T0) - 2400)

## 3. Load the Phase-1 checkpoint and tokenizer

In [3]:
CK_DIR = None

found = glob.glob("/kaggle/input/**/final_params.pkl", recursive=True)
print("checkpoints found:", found)
CK_DIR = CK_DIR or os.path.dirname(found[0])

ck = pickle.load(open(f"{CK_DIR}/final_params.pkl", "rb"))
P0 = tmap(jnp.asarray, ck["params"])
cfg = ck["cfg"]
KINDS = tuple(cfg["kinds"])
print("variant:", ck["variant"], "| block kinds:", "".join(KINDS))
assert KINDS and "blocks" in P0 and len(KINDS) == len(P0["blocks"])

tk = json.load(open(f"{CK_DIR}/bpe_legal.json"))

V, D = P0["emb"].shape
HD = 64
H = D // HD
KV = cfg["kvh"]
R = H // KV
print(f"vocab {V}, dim {D}, blocks {len(P0['blocks'])}, backbone params {sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(P0)) / 1e6:.1f}M")

checkpoints found: ['/kaggle/input/datasets/nirbhaykachhatiya/phase3m2/final_params.pkl']
variant: ours_glru | block kinds: RRRRRRRRRRRR
vocab 16384, dim 768, blocks 12, backbone params 97.6M


## 4. Sentence tokenizer (byte-level BPE from Phase 1)

In [4]:
PAT = regex.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+""")

byte_list = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
char_list = byte_list[:]
extra = 0
for b in range(256):
    if b not in byte_list:
        byte_list.append(b)
        char_list.append(256 + extra)
        extra += 1
B2U = {b: chr(c) for b, c in zip(byte_list, char_list)}

stoi = {s: i for i, s in enumerate(tk["itos"])}
rank = {tuple(m): i for i, m in enumerate(tk["merges"])}
bpe_cache = {}


def bpe(word):
    if word not in bpe_cache:
        syms = [B2U[b] for b in word.encode()]
        while len(syms) > 1:
            best_rank, best_i = min((rank.get(p, BIG), i) for i, p in enumerate(zip(syms, syms[1:])))
            if best_rank == BIG:
                break
            syms[best_i:best_i + 2] = [syms[best_i] + syms[best_i + 1]]
        bpe_cache[word] = [stoi[s] for s in syms]
    return bpe_cache[word]


def encode_sentence(sentence):
    ids = []
    for w in PAT.findall(" " + " ".join(str(sentence).split())):
        ids.extend(bpe(w))
    return ids[:SENT_TOKENS - 1] + [EOS]

## 5. Load Phase-3 data (CL and IT train / dev / test) and the label vocabulary

In [5]:
def hf_file(name):
    return hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_3_RR/{name}", repo_type="dataset", local_dir="/tmp/hf")


def load_docs(name):
    data = json.load(open(hf_file(name)))
    if isinstance(data, dict):
        data = [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v} for k, v in data.items()]
    return data


def parse_labels(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if all(str(k).isdigit() for k in raw):
        return [raw[k] for k in sorted(raw, key=int)]
    if all(isinstance(v, int) for v in raw.values()):
        return [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
    return list(raw)


LABELS = parse_labels(json.load(open(hf_file("Train/label_vocab.json"))))
K = len(LABELS)
LAB_ID = {l: i for i, l in enumerate(LABELS)}
print(K, "labels:", LABELS)

DOCS = {}
for dom in ("CL", "IT"):
    DOCS[dom] = {}
    for split in ("train", "dev", "test"):
        folder = "Test" if split == "test" else "Train"
        raw = load_docs(f"{folder}/{dom}_{split}.json")
        DOCS[dom][split] = [
            {
                "id": str(d["id"]),
                "sent_ids": [encode_sentence(s) for s in d["text"]],
                "y": np.array([LAB_ID[l] for l in d["labels"]], np.int32) if "labels" in d else None,
            }
            for d in raw
        ]
        n_sent = sum(len(d["sent_ids"]) for d in DOCS[dom][split])
        n_tok = sum(len(s) for d in DOCS[dom][split] for s in d["sent_ids"])
        print(f"{dom} {split}: {len(DOCS[dom][split])} docs, {n_sent} sentences, {n_tok / max(n_sent, 1):.1f} tokens/sentence")

for dom in ("CL", "IT"):
    y = np.concatenate([d["y"] for d in DOCS[dom]["train"]])
    print(dom, "train label counts:", dict(zip(LABELS, np.bincount(y, minlength=K).tolist())))

ALL_TRAIN = DOCS["CL"]["train"] + DOCS["IT"]["train"]
counts = np.bincount(np.concatenate([d["y"] for d in ALL_TRAIN]), minlength=K).astype(np.float64)
CLASS_W = np.where(counts > 0, np.clip((np.median(counts[counts > 0]) / np.maximum(counts, 1)) ** 0.5, 0.25, 4.0), 1.0).astype(np.float32)
print("class weights:", dict(zip(LABELS, CLASS_W.round(2).tolist())))

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

13 labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL train: 40 docs, 10593 sentences, 41.4 tokens/sentence


CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL dev: 5 docs, 1396 sentences, 43.3 tokens/sentence


... (4 outputs left out)

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT dev: 5 docs, 746 sentences, 40.0 tokens/sentence


IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

IT test: 5 docs, 858 sentences, 44.3 tokens/sentence
CL train label counts: {'Fact': 2979, 'Issue': 17, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'Statute': 367, 'Dissent': 284, 'PrecedentReliedUpon': 1305, 'PrecedentNotReliedUpon': 209, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 438, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'None': 53}
IT train label counts: {'Fact': 1119, 'Issue': 90, 'ArgumentPetitioner': 637, 'ArgumentRespondent': 352, 'Statute': 732, 'Dissent': 0, 'PrecedentReliedUpon': 640, 'PrecedentNotReliedUpon': 360, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 436, 'RatioOfTheDecision': 1445, 'RulingByPresentCourt': 194, 'None': 244}
class weights: {'Fact': 0.46000000834465027, 'Issue': 2.859999895095825, 'ArgumentPetitioner': 0.7400000095367432, 'ArgumentRespondent': 0.8100000023841858, 'Statute': 0.8899999856948853, 'Dissent': 1.75, 'PrecedentReliedUpon': 0.6700000166893005, 'PrecedentNotReliedUpon': 1.2400000095367432, 'PrecedentOverruled

## 6. Sentence-only and context-aware windows

In [6]:
def make_windows(sent_ids, mode):
    n = len(sent_ids)
    if mode == "sentence":
        return [(i, i + 1) for i in range(n)]
    out, s = [], 0
    while s < n:
        e, tok = s, 0
        while e < n and e - s < MAX_SENTS and tok + len(sent_ids[e]) <= CONTEXT_TOKENS:
            tok += len(sent_ids[e])
            e += 1
        e = max(e, s + 1)
        out.append((s, e))
        if e >= n:
            break
        s += max(1, (e - s) // 2)
    return out


def build_set(docs, mode):
    T = SENT_TOKENS if mode == "sentence" else CONTEXT_TOKENS
    S = 1 if mode == "sentence" else MAX_SENTS
    ids, sidx, ys, meta = [], [], [], []
    for di, d in enumerate(docs):
        for s, e in make_windows(d["sent_ids"], mode):
            row = np.zeros(T, np.int32)
            si = np.full(T, -1, np.int32)
            y = np.full(S, -1, np.int32)
            pos = 0
            for j in range(s, e):
                seg = d["sent_ids"][j]
                row[pos:pos + len(seg)] = seg
                si[pos:pos + len(seg)] = j - s
                pos += len(seg)
                if d["y"] is not None:
                    y[j - s] = d["y"][j]
            ids.append(row)
            sidx.append(si)
            ys.append(y)
            meta.append((di, s, e))
    return {"ids": np.stack(ids), "sidx": np.stack(sidx), "y": np.stack(ys), "meta": meta}


SETS = {}


def get_sets(mode):
    if mode not in SETS:
        SETS[mode] = {
            "train": build_set(ALL_TRAIN, mode),
            "dev": {dom: build_set(DOCS[dom]["dev"], mode) for dom in ("CL", "IT")},
            "test": {dom: build_set(DOCS[dom]["test"], mode) for dom in ("CL", "IT")},
        }
        print(mode, "windows: train", len(SETS[mode]["train"]["ids"]), "| dev",
              {d: len(v["ids"]) for d, v in SETS[mode]["dev"].items()})
    return SETS[mode]


for m in ("context", "sentence"):
    get_sets(m)

context windows: train 3000 | dev {'CL': 258, 'IT': 126}
sentence windows: train 16845 | dev {'CL': 1396, 'IT': 746}


## 7. Model: Phase-1 gated linear-recurrent backbone (causal) + adapters + bidirectional attention head + sentence pooling

In [7]:
inv_freq = 1 / (10000 ** (np.arange(0, HD, 2) / HD))
angles = np.arange(1280)[:, None] * inv_freq
COS = jnp.asarray(np.cos(angles), jnp.float32)
SIN = jnp.asarray(np.sin(angles), jnp.float32)
CLASS_W_J = jnp.asarray(CLASS_W)
LOGIT_SENTS = {SENT_TOKENS: 1, CONTEXT_TOKENS: MAX_SENTS}
ADAPTED = ("wq", "wk", "wv", "wo", "wx", "wg", "wa", "w1", "w2", "w3")
HEAD_KEYS = ("head", "nh", "cls", "cb")
PEFT_SUFFIX = ("_ma", "_mb", "_la", "_lb")


def rmsn(x, w):
    return x * lax.rsqrt((x * x).mean(-1, keepdims=True) + 1e-6) * w


def rope(x, cos, sin):
    x1, x2 = x[..., :HD // 2], x[..., HD // 2:]
    return jnp.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1)


def lin(p, name, x):
    y = x @ p[name]
    if name + "_mb" in p:
        # LoRMA+: multiplicative update acting on the output of the frozen weight
        y = y + PEFT_SCALE * (y @ p[name + "_ma"]) @ p[name + "_mb"]
    elif name + "_lb" in p:
        # LoRA: additive update acting on the input
        y = y + PEFT_SCALE * (x @ p[name + "_la"]) @ p[name + "_lb"]
    return y


def mlp(p, x):
    h = rmsn(x, p["n2"])
    return x + lin(p, "w2", jax.nn.silu(lin(p, "w1", h)) * lin(p, "w3", h))


def recurrent_block(p, x):
    Bsz, T, _ = x.shape
    h = rmsn(x, p["n1"])
    xv = lin(p, "wx", h)
    k_conv = p["conv"].shape[0]
    xp = jnp.concatenate([jnp.zeros((Bsz, k_conv - 1, xv.shape[-1])), xv], 1)
    u = sum(xp[:, j:j + T] * p["conv"][j] for j in range(k_conv))
    a = jax.nn.sigmoid(lin(p, "wa", h) + p["ab"])
    b = (1.0 - a) * u
    _, s = lax.associative_scan(lambda l, r: (l[0] * r[0], r[0] * l[1] + r[1]), (a, b), axis=1)
    x = x + lin(p, "wo", s * jax.nn.silu(lin(p, "wg", h)))
    return mlp(p, x)


def attention_block(p, x, cos, sin, causal, valid=None):
    Bsz, T, _ = x.shape
    h = rmsn(x, p["n1"])
    q = rope(rmsn(lin(p, "wq", h).reshape(Bsz, T, H, HD), p["qn"]), cos, sin)
    k = rope(rmsn(lin(p, "wk", h).reshape(Bsz, T, KV, HD), p["kn"]), cos, sin)
    v = lin(p, "wv", h).reshape(Bsz, T, KV, HD)

    scores = jnp.einsum("btgrd,bsgd->bgrts", q.reshape(Bsz, T, KV, R, HD), k) * HD ** -0.5
    dist = jnp.arange(T)[:, None] - jnp.arange(T)[None]
    mask = (dist >= 0) if causal else (jnp.abs(dist) < BIG)
    mask = mask[None] & (valid[:, None, :] if valid is not None else True)

    attn = jax.nn.softmax(jnp.where(mask[:, None, None], scores, -1e30), -1)
    out = jnp.einsum("bgrts,bsgd->btgrd", attn, v).reshape(Bsz, T, H * HD)
    x = x + lin(p, "wo", out)
    return mlp(p, x)


def logits_fn(P, kinds, ids, sidx):
    T = ids.shape[1]
    n_sent = LOGIT_SENTS[T]
    cos, sin = COS[:T][None, :, None, :], SIN[:T][None, :, None, :]
    valid = sidx >= 0

    x = P["emb"][ids]
    for kind, p in zip(kinds, P["blocks"]):
        x = recurrent_block(p, x) if kind == "R" else attention_block(p, x, cos, sin, True)
    x = rmsn(x, P["nf"])
    x = attention_block(P["head"], x, cos, sin, False, valid)

    onehot = jax.nn.one_hot(sidx, n_sent)
    pooled = jnp.einsum("bts,btd->bsd", onehot, x) / jnp.maximum(onehot.sum(1), 1.0)[..., None]
    return rmsn(pooled, P["nh"]) @ P["cls"] + P["cb"]


def init_head(key):
    ks = jax.random.split(key, 8)
    n = lambda i, shape: jax.random.normal(ks[i], shape) * 0.02
    ff = cfg["ff"]
    return {
        "wq": n(0, (D, H * HD)), "wk": n(1, (D, KV * HD)), "wv": n(2, (D, KV * HD)), "wo": n(3, (H * HD, D)),
        "n1": jnp.ones(D), "n2": jnp.ones(D), "qn": jnp.ones(HD), "kn": jnp.ones(HD),
        "w1": n(4, (D, ff)), "w3": n(5, (D, ff)), "w2": n(6, (ff, D)),
    }


def build_model(blocks, adapter, key):
    blocks = [dict(b) for b in blocks]
    counter = 0
    if adapter in ("lorma", "lora"):
        for p in blocks:
            for name in ADAPTED:
                if name not in p:
                    continue
                d_in, d_out = p[name].shape
                k = jax.random.fold_in(key, counter)
                counter += 1
                if adapter == "lorma":
                    p[name + "_ma"] = jax.random.normal(k, (d_out, RANK)) * 0.02
                    p[name + "_mb"] = jnp.zeros((RANK, d_out))
                else:
                    p[name + "_la"] = jax.random.normal(k, (d_in, RANK)) * 0.02
                    p[name + "_lb"] = jnp.zeros((RANK, d_out))
    return {
        "emb": P0["emb"],
        "blocks": blocks,
        "nf": P0["nf"],
        "head": init_head(jax.random.fold_in(key, 10_000)),
        "nh": jnp.ones(D),
        "cls": jax.random.normal(jax.random.fold_in(key, 10_001), (D, K)) * 0.02,
        "cb": jnp.zeros(K),
    }


def trainable_mask(P, method):
    if method == "full":
        return jax.tree_util.tree_map_with_path(lambda path, x: True, P)
    return jax.tree_util.tree_map_with_path(
        lambda path, x: path[0].key in HEAD_KEYS or str(path[-1].key).endswith(PEFT_SUFFIX), P)


def count_params(P, mask=None):
    leaves = jax.tree_util.tree_leaves(P)
    if mask is None:
        return sum(int(np.prod(x.shape)) for x in leaves)
    return sum(int(np.prod(x.shape)) for x, tr in zip(leaves, jax.tree_util.tree_leaves(mask)) if tr)


logp_fn = jax.jit(lambda P, kinds, ids, sidx: jax.nn.log_softmax(logits_fn(P, kinds, ids, sidx)), static_argnums=1)
logits_jit = jax.jit(logits_fn, static_argnums=1)

## 8. Prediction (one label per sentence) and Macro-F1 / Precision / Recall

In [8]:
def predict_scores(P, kinds, data, docs):
    T = data["ids"].shape[1]
    pb = PRED_TOKENS // T
    acc = [np.zeros((len(d["sent_ids"]), K), np.float32) for d in docs]
    cnt = [np.zeros(len(d["sent_ids"]), np.float32) for d in docs]
    for i in range(0, len(data["ids"]), pb):
        ids, sidx = data["ids"][i:i + pb], data["sidx"][i:i + pb]
        pad = pb - len(ids)
        if pad:
            ids = np.concatenate([ids, np.zeros((pad, T), np.int32)])
            sidx = np.concatenate([sidx, np.full((pad, T), -1, np.int32)])
        lp = np.asarray(logp_fn(P, kinds, ids, sidx))
        for j, (di, s, e) in enumerate(data["meta"][i:i + pb]):
            acc[di][s:e] += lp[j, :e - s]
            cnt[di][s:e] += 1
    return [a / c[:, None] for a, c in zip(acc, cnt)]


def macro_scores(gold, pred):
    present = sorted(set(gold.tolist()) | set(pred.tolist()))
    f1_all = np.zeros(K)
    precs, recs, f1s = [], [], []
    for k in present:
        tp = int(((gold == k) & (pred == k)).sum())
        fp = int(((gold != k) & (pred == k)).sum())
        fn = int(((gold == k) & (pred != k)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        precs.append(pr)
        recs.append(rc)
        f1s.append(f1)
        f1_all[k] = f1
    return float(np.mean(f1s)), float(np.mean(precs)), float(np.mean(recs)), f1_all


def dev_predictions(P, kinds, mode):
    sets = get_sets(mode)
    out = {}
    for dom in ("CL", "IT"):
        scores = predict_scores(P, kinds, sets["dev"][dom], DOCS[dom]["dev"])
        pred = np.concatenate([s.argmax(1) for s in scores])
        gold = np.concatenate([d["y"] for d in DOCS[dom]["dev"]])
        out[dom] = (gold, pred)
    return out


def evaluate(P, kinds, mode):
    out = dev_predictions(P, kinds, mode)
    res = {dom: macro_scores(*out[dom]) for dom in out}
    pooled = macro_scores(np.concatenate([out[d][0] for d in out]), np.concatenate([out[d][1] for d in out]))
    return (res["CL"][0] + res["IT"][0]) / 2, res, pooled


def save_submission(name, P, kinds, mode):
    t_start = time.time()
    sets = get_sets(mode)
    result = {}
    for dom in ("CL", "IT"):
        scores = predict_scores(P, kinds, sets["test"][dom], DOCS[dom]["test"])
        for d, s in zip(DOCS[dom]["test"], scores):
            assert len(s) == len(d["sent_ids"])
            if d["id"] in result:
                print("warning: duplicate document id across domains:", d["id"])
            result[d["id"]] = [LABELS[i] for i in s.argmax(1)]
    elapsed = time.time() - t_start
    out_dir = f"/kaggle/working/{name}"
    os.makedirs(out_dir, exist_ok=True)
    json.dump(result, open(f"{out_dir}/prediction.json", "w"))
    with zipfile.ZipFile(f"/kaggle/working/{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{out_dir}/prediction.json", "prediction.json")
    print(f"{name}: {len(result)} documents, {sum(len(v) for v in result.values())} sentences, inference {elapsed:.0f}s")
    return elapsed

## 9. Training loop (class-weighted CE, optional distillation loss, frozen weights via stop-gradient, early stopping)

In [9]:
def make_step(kinds, mask, lr_mult):
    def loss_fn(P, ids, sidx, y, teacher_logits, alpha):
        Pe = tmap(lambda p, tr: p if tr else lax.stop_gradient(p), P, mask)
        logits = logits_fn(Pe, kinds, ids, sidx)
        logp = jax.nn.log_softmax(logits)
        keep = y >= 0
        yy = jnp.maximum(y, 0)
        w = CLASS_W_J[yy] * keep
        ce = (-jnp.take_along_axis(logp, yy[..., None], -1)[..., 0] * w).sum() / w.sum()

        t_logp = jax.nn.log_softmax(teacher_logits / KD_TEMP)
        s_logp = jax.nn.log_softmax(logits / KD_TEMP)
        kd = ((jnp.exp(t_logp) * (t_logp - s_logp)).sum(-1) * keep).sum() / jnp.maximum(keep.sum(), 1) * KD_TEMP ** 2
        return (1 - alpha) * ce + alpha * kd

    @jax.jit
    def step(P, m, v, t, lr, ids, sidx, y, teacher_logits, alpha):
        loss, g = jax.value_and_grad(loss_fn)(P, ids, sidx, y, teacher_logits, alpha)
        gnorm = jnp.sqrt(sum((x ** 2).sum() for x in jax.tree_util.tree_leaves(g)))
        g = tmap(lambda x: x * jnp.minimum(1.0, 1.0 / (gnorm + 1e-6)), g)
        m = tmap(lambda a, b: 0.9 * a + 0.1 * b, m, g)
        v = tmap(lambda a, b: 0.999 * a + 0.001 * b * b, v, g)

        def update(p, a, b, mult):
            adam = (a / (1 - 0.9 ** t)) / (jnp.sqrt(b / (1 - 0.999 ** t)) + 1e-8)
            return p - lr * mult * (adam + 0.01 * p)

        return tmap(update, P, m, v, lr_mult), m, v, loss

    return step


def train(name, P, kinds, mode, method, minutes, peak_lr, head_mult, teacher_logits=None, alpha=0.0):
    mask = trainable_mask(P, method)
    lr_mult = jax.tree_util.tree_map_with_path(
        lambda path, x, tr: float(tr) * (head_mult if path[0].key in HEAD_KEYS else 1.0), P, mask)
    step = make_step(kinds, mask, lr_mult)
    m = tmap(jnp.zeros_like, P)
    v = tmap(jnp.zeros_like, P)
    data = get_sets(mode)["train"]
    T = data["ids"].shape[1]
    batch = TRAIN_TOKENS // T
    S = data["y"].shape[1]
    zeros = jnp.zeros((batch, S, K))
    steps_per_epoch = max(len(data["ids"]) // batch, 1)
    seconds = budget(minutes)
    print(f"[{name}] mode {mode} | trainable {count_params(P, mask) / 1e6:.2f}M of {count_params(P) / 1e6:.1f}M | {steps_per_epoch} steps/epoch")

    t, total, done, bad = 0, None, False, 0
    best = (-1, None, None)
    t_start = time.time()

    while not done:
        perm = np.random.permutation(len(data["ids"]))
        for i in range(steps_per_epoch):
            ix = perm[i * batch:(i + 1) * batch]
            t += 1
            lr = peak_lr * min(1, t / WARMUP_STEPS)
            if total:
                lr *= 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1, t / total)))
            tl = jnp.asarray(teacher_logits[ix], jnp.float32) if teacher_logits is not None else zeros
            P, m, v, loss = step(P, m, v, jnp.float32(t), lr, data["ids"][ix], data["sidx"][ix], data["y"][ix], tl, jnp.float32(alpha))
            if t == 5:
                loss.block_until_ready()
                t_bench = time.time()
            if t == 25:
                loss.block_until_ready()
                sec_per_step = (time.time() - t_bench) / 20
                total = min(MAX_EPOCHS * steps_per_epoch, int(0.9 * seconds / sec_per_step))
                print(f"[{name}] {sec_per_step:.2f}s/step -> {total} steps ({total / steps_per_epoch:.1f} epochs)")
            if (total and t >= total) or time.time() - T0 > MAX_SECONDS:
                done = True
                break

        mean_f1, res, pooled = evaluate(P, kinds, mode)
        print(f"[{name}] step {t} loss {float(loss):.4f} | dev macro-F1 CL {res['CL'][0]:.4f} IT {res['IT'][0]:.4f} mean {mean_f1:.4f} "
              f"| {(time.time() - t_start) / 60:.1f} min", flush=True)
        if mean_f1 > best[0]:
            best, bad = (mean_f1, P, (res, pooled)), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{name}] early stopping")
                done = True

    return best, time.time() - t_start

## 10. Experiment 1 — LoRMA+ (rank 16), context-aware windows (also the distillation teacher)

In [10]:
RESULTS = []

model = build_model(P0["blocks"], "lorma", jax.random.PRNGKey(0))
n_trainable = count_params(model, trainable_mask(model, "peft"))
(best_f1, teacher, (res, pooled)), train_secs = train("lorma_ctx", model, KINDS, "context", "peft", LORMA_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_glru_lorma", teacher, KINDS, "context")
RESULTS.append(("lorma_ctx", n_trainable, count_params(teacher), best_f1, res, pooled, train_secs, infer_secs))
del model

[lorma_ctx] mode context | trainable 9.35M of 106.9M | 375 steps/epoch
[lorma_ctx] 0.53s/step -> 8157 steps (21.8 epochs)
[lorma_ctx] step 375 loss 0.9026 | dev macro-F1 CL 0.3930 IT 0.5041 mean 0.4486 | 4.6 min
[lorma_ctx] step 750 loss 1.3376 | dev macro-F1 CL 0.3843 IT 0.5009 mean 0.4426 | 8.8 min
[lorma_ctx] step 1125 loss 0.1899 | dev macro-F1 CL 0.4487 IT 0.5486 mean 0.4986 | 13.0 min
[lorma_ctx] step 1500 loss 0.1685 | dev macro-F1 CL 0.4764 IT 0.5585 mean 0.5174 | 17.2 min
[lorma_ctx] step 1875 loss 0.3427 | dev macro-F1 CL 0.3675 IT 0.5303 mean 0.4489 | 21.5 min
[lorma_ctx] step 2250 loss 0.0120 | dev macro-F1 CL 0.4157 IT 0.5698 mean 0.4927 | 25.7 min
[lorma_ctx] step 2625 loss 0.0432 | dev macro-F1 CL 0.3792 IT 0.5692 mean 0.4742 | 29.9 min
[lorma_ctx] step 3000 loss 0.0238 | dev macro-F1 CL 0.4007 IT 0.5197 mean 0.4602 | 34.1 min
[lorma_ctx] step 3375 loss 0.0138 | dev macro-F1 CL 0.4168 IT 0.5690 mean 0.4929 | 38.3 min
[lorma_ctx] step 3750 loss 0.0012 | dev macro-F1 CL 0.

## 11. Experiment 2 — Knowledge distillation (half-depth student, teacher = Experiment 1)

In [11]:
data = get_sets("context")["train"]
teacher_logits = np.concatenate([
    np.asarray(logits_jit(teacher, KINDS, data["ids"][i:i + 16], data["sidx"][i:i + 16]))
    for i in range(0, len(data["ids"]), 16)
]).astype(np.float16)
print("teacher logits:", teacher_logits.shape)

student_idx = np.arange(0, len(KINDS), 2)
STUDENT_KINDS = tuple(KINDS[i] for i in student_idx)
print("teacher blocks:", "".join(KINDS), "-> student blocks:", "".join(STUDENT_KINDS))
student = build_model([P0["blocks"][i] for i in student_idx], None, jax.random.PRNGKey(1))
n_trainable = count_params(student)
(best_f1, best_P, (res, pooled)), train_secs = train(
    "kd_student", student, STUDENT_KINDS, "context", "full", KD_MINUTES, 5e-5, 10.0, teacher_logits=teacher_logits, alpha=KD_ALPHA)
infer_secs = save_submission("p3_glru_kd", best_P, STUDENT_KINDS, "context")
RESULTS.append(("kd_student", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del best_P, student, teacher_logits

teacher logits: (3000, 48, 13)
teacher blocks: RRRRRRRRRRRR -> student blocks: RRRRRR
[kd_student] mode context | trainable 61.39M of 61.4M | 375 steps/epoch
[kd_student] 0.40s/step -> 9436 steps (25.2 epochs)
[kd_student] step 375 loss 0.6568 | dev macro-F1 CL 0.3820 IT 0.5229 mean 0.4525 | 2.9 min
[kd_student] step 750 loss 0.2672 | dev macro-F1 CL 0.4164 IT 0.5248 mean 0.4706 | 5.6 min
[kd_student] step 1125 loss 0.1529 | dev macro-F1 CL 0.4096 IT 0.5684 mean 0.4890 | 8.2 min
[kd_student] step 1500 loss 0.2459 | dev macro-F1 CL 0.4085 IT 0.5047 mean 0.4566 | 10.9 min
[kd_student] step 1875 loss 0.2460 | dev macro-F1 CL 0.4244 IT 0.5386 mean 0.4815 | 13.5 min
[kd_student] step 2250 loss 0.1620 | dev macro-F1 CL 0.4040 IT 0.5297 mean 0.4669 | 16.2 min
[kd_student] step 2625 loss 0.1080 | dev macro-F1 CL 0.4338 IT 0.5336 mean 0.4837 | 18.8 min
[kd_student] step 3000 loss 0.1207 | dev macro-F1 CL 0.4331 IT 0.5504 mean 0.4917 | 21.5 min
[kd_student] step 3375 loss 0.1387 | dev macro-F1 C

## 12. Experiment 3 — LoRA (rank 16, additive), context-aware windows

In [12]:
model = build_model(P0["blocks"], "lora", jax.random.PRNGKey(0))
n_trainable = count_params(model, trainable_mask(model, "peft"))
(best_f1, best_P, (res, pooled)), train_secs = train("lora_ctx", model, KINDS, "context", "peft", LORA_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_glru_lora", best_P, KINDS, "context")
RESULTS.append(("lora_ctx", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del model, best_P

[lora_ctx] mode context | trainable 9.11M of 106.7M | 375 steps/epoch
[lora_ctx] 0.63s/step -> 5995 steps (16.0 epochs)
[lora_ctx] step 375 loss 1.2423 | dev macro-F1 CL 0.4026 IT 0.5214 mean 0.4620 | 4.7 min
[lora_ctx] step 750 loss 0.8956 | dev macro-F1 CL 0.3125 IT 0.4863 mean 0.3994 | 8.8 min
[lora_ctx] step 1125 loss 0.1882 | dev macro-F1 CL 0.3781 IT 0.4911 mean 0.4346 | 13.0 min
[lora_ctx] step 1500 loss 0.2398 | dev macro-F1 CL 0.4832 IT 0.5390 mean 0.5111 | 17.2 min
[lora_ctx] step 1875 loss 0.0217 | dev macro-F1 CL 0.3932 IT 0.5425 mean 0.4678 | 21.4 min
[lora_ctx] step 2250 loss 0.0088 | dev macro-F1 CL 0.4252 IT 0.5612 mean 0.4932 | 25.6 min
[lora_ctx] step 2625 loss 0.0004 | dev macro-F1 CL 0.4801 IT 0.5276 mean 0.5039 | 29.8 min
[lora_ctx] step 3000 loss 0.0008 | dev macro-F1 CL 0.4130 IT 0.5660 mean 0.4895 | 33.9 min
[lora_ctx] step 3375 loss 0.0002 | dev macro-F1 CL 0.4611 IT 0.5685 mean 0.5148 | 38.1 min
[lora_ctx] step 3750 loss 0.0001 | dev macro-F1 CL 0.4589 IT 0.56

## 13. Experiment 4 — LoRMA+ (rank 16), sentence-only input (context ablation)

In [13]:
model = build_model(P0["blocks"], "lorma", jax.random.PRNGKey(0))
n_trainable = count_params(model, trainable_mask(model, "peft"))
(best_f1, best_P, (res, pooled)), train_secs = train("lorma_sent", model, KINDS, "sentence", "peft", SENTENCE_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_glru_lorma_sentence", best_P, KINDS, "sentence")
RESULTS.append(("lorma_sent", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del model, best_P

[lorma_sent] mode sentence | trainable 9.35M of 106.9M | 526 steps/epoch
[lorma_sent] 0.57s/step -> 4755 steps (9.0 epochs)
[lorma_sent] step 526 loss 1.4878 | dev macro-F1 CL 0.2597 IT 0.4368 mean 0.3483 | 5.8 min
[lorma_sent] step 1052 loss 1.1385 | dev macro-F1 CL 0.2542 IT 0.4146 mean 0.3344 | 11.0 min
[lorma_sent] step 1578 loss 0.6381 | dev macro-F1 CL 0.2633 IT 0.3887 mean 0.3260 | 16.3 min
[lorma_sent] step 2104 loss 0.1814 | dev macro-F1 CL 0.2554 IT 0.4369 mean 0.3461 | 21.6 min
[lorma_sent] step 2630 loss 0.4793 | dev macro-F1 CL 0.2465 IT 0.3967 mean 0.3216 | 26.9 min
[lorma_sent] step 3156 loss 0.0039 | dev macro-F1 CL 0.2593 IT 0.3912 mean 0.3252 | 32.2 min
[lorma_sent] step 3682 loss 0.0028 | dev macro-F1 CL 0.2620 IT 0.4037 mean 0.3328 | 37.5 min
[lorma_sent] early stopping
p3_glru_lorma_sentence: 10 documents, 2197 sentences, inference 19s


## 14. Results summary

In [14]:
print(f"{'experiment':12s}{'trainable(M)':>13s}{'total(M)':>10s}{'CL F1':>8s}{'CL P':>8s}{'CL R':>8s}{'IT F1':>8s}{'IT P':>8s}{'IT R':>8s}{'mean F1':>9s}{'train(min)':>11s}{'infer(s)':>9s}")
for name, n_tr, n_tot, mean_f1, res, pooled, tr, inf in RESULTS:
    cl, it = res["CL"], res["IT"]
    print(f"{name:12s}{n_tr / 1e6:13.2f}{n_tot / 1e6:10.1f}{cl[0]:8.4f}{cl[1]:8.4f}{cl[2]:8.4f}{it[0]:8.4f}{it[1]:8.4f}{it[2]:8.4f}{mean_f1:9.4f}{tr / 60:11.1f}{inf:9.0f}")

print()
print(f"{'label (pooled F1)':26s}" + "".join(f"{r[0]:>12s}" for r in RESULTS))
for k, label in enumerate(LABELS):
    print(f"{label:26s}" + "".join(f"{r[5][3][k]:12.4f}" for r in RESULTS))

by_name = {r[0]: r for r in RESULTS}
ctx, sent = by_name["lorma_ctx"], by_name["lorma_sent"]
print(f"\ncontext effect (LoRMA+ mean dev macro-F1): context {ctx[3]:.4f} vs sentence-only {sent[3]:.4f} -> {ctx[3] - sent[3]:+.4f}")
print(f"multiplicative vs additive (context): LoRMA+ {ctx[3]:.4f} vs LoRA {by_name['lora_ctx'][3]:.4f} -> {ctx[3] - by_name['lora_ctx'][3]:+.4f}")
print(f"distillation: teacher {ctx[3]:.4f} ({ctx[2] / 1e6:.1f}M) vs student {by_name['kd_student'][3]:.4f} ({by_name['kd_student'][2] / 1e6:.1f}M)")
print(f"total time {(time.time() - T0) / 3600:.2f} h")

experiment   trainable(M)  total(M)   CL F1    CL P    CL R   IT F1    IT P    IT R  mean F1 train(min) infer(s)
lorma_ctx            9.35     106.9  0.4764  0.4740  0.5371  0.5585  0.6090  0.5380   0.5174       42.5       14
kd_student          61.39      61.4  0.4744  0.5019  0.5273  0.5454  0.6429  0.5082   0.5099       45.2        7
lora_ctx             9.11     106.7  0.4691  0.4844  0.5012  0.5705  0.6612  0.5318   0.5198       67.4       14
lorma_sent           9.35     106.9  0.2597  0.2589  0.2798  0.4368  0.5137  0.4129   0.3483       37.5       19

label (pooled F1)            lorma_ctx  kd_student    lora_ctx  lorma_sent
Fact                            0.6635      0.6950      0.7368      0.5579
Issue                           0.7857      0.6897      0.6923      0.5217
ArgumentPetitioner              0.6829      0.5335      0.5120      0.4435
ArgumentRespondent              0.7004      0.5714      0.6087      0.2663
Statute                         0.8291      0.7881      0.8

## 15. Error analysis — per-domain F1 and most frequent confusions (LoRMA+ context model)

In [15]:
dev = dev_predictions(teacher, KINDS, "context")
print(f"{'label':26s}{'CL F1':>9s}{'IT F1':>9s}{'CL gold':>9s}{'IT gold':>9s}")
for k, label in enumerate(LABELS):
    row = [by_name["lorma_ctx"][4][dom][3][k] for dom in ("CL", "IT")]
    n_gold = [int((dev[dom][0] == k).sum()) for dom in ("CL", "IT")]
    print(f"{label:26s}{row[0]:9.4f}{row[1]:9.4f}{n_gold[0]:9d}{n_gold[1]:9d}")

for dom, (gold, pred) in dev.items():
    cm = np.zeros((K, K), int)
    np.add.at(cm, (gold, pred), 1)
    off = sorted(((cm[i, j], LABELS[i], LABELS[j]) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0), reverse=True)
    print(f"\n{dom}: most frequent confusions (gold -> predicted)")
    for c, a, b in off[:10]:
        print(f"  {a:24s} -> {b:24s} {c:5d}")

label                         CL F1    IT F1  CL gold  IT gold
Fact                         0.5902   0.7640      249      195
Issue                        0.4000   0.8696        3       12
ArgumentPetitioner           0.6973   0.6055      344       56
ArgumentRespondent           0.7317   0.4762      236       36
Statute                      0.9016   0.7500       58       62
Dissent                      0.0000   0.0000        0        0
PrecedentReliedUpon          0.3988   0.5766      142      112
PrecedentNotReliedUpon       0.0000   0.2078        0       50
PrecedentOverruled           0.0000   0.0000        0        0
RulingByLowerCourt           0.2667   0.5000       56       46
RatioOfTheDecision           0.5464   0.5260      288      115
RulingByPresentCourt         0.5965   0.5000       18       31
None                         0.1111   0.3673        2       31

CL: most frequent confusions (gold -> predicted)
  RatioOfTheDecision       -> PrecedentReliedUpon         61
  Fact 